# SenGuard-MSC kernel G: independent test of the score forecast
Needs kernel F attached. The frozen model is read from its output.

In [ ]:
# Restores the frozen outputs of kernel A at their original absolute paths, so hash checks stay meaningful.
import shutil
from pathlib import Path
hits = sorted(Path('/kaggle/input').rglob('frozen_model.json'))
if not hits:
    raise FileNotFoundError('Kernel A output is not attached: no frozen_model.json under /kaggle/input.')
src = hits[0].parents[1]
dst = Path('/kaggle/working/senguard_msc')
if not dst.exists():
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns('.git', '.venv_check'))
print('Restored', src, '->', dst, '|', sorted(p.name for p in dst.iterdir()))

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'], capture_output=True, text=True).returncode)

SenGuard-MSC · Notebook 06 — Protocol amendment: more MSC cohorts, head-to-head, robustness, drift clock

Written after G3 was seen, so this is an amendment and is labelled as one. Cohorts below were chosen from metadata before any of them was scored. The amendment is written to disk and hashed before any expression file is read.

In [ ]:
import gc, gzip, io, json, os, re, subprocess, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, requests
ROOT_GUESS = Path('/kaggle/working/senguard_msc') if Path('/kaggle/working/senguard_msc').exists() else (Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()) / 'senguard_msc'
sys.path.insert(0, str(ROOT_GUESS / 'src'))
import senguard as sg
from senguard import ROOT, PROC, RESULTS, REPORTS, MANIFESTS, EXT, DEV, ASC, BM
from scipy import stats
import matplotlib.pyplot as plt
sg.set_fig_style()
np.random.seed(sg.SEED)
warnings.filterwarnings('ignore', category=FutureWarning)
RAWX = ROOT / 'data/raw_extended'
RAWX.mkdir(parents=True, exist_ok=True)
FREEZE = json.loads((RESULTS / 'frozen_model.json').read_text())
for name, art in FREEZE['artifacts'].items():
    if sg.sha256(art['path']) != art['sha256']:
        raise RuntimeError(f'{name} changed after the freeze')
print('Frozen artifacts verified. Model weights and published SenCat weights are unchanged.')

AMENDMENT = """# Analysis protocol amendment 1

Fixed on 2026-11-01, after G3 results on GSE309666 and GSE178514 were seen, and before any cohort below was scored.
Inclusion came from GEO metadata only. Published and refit weights stay frozen.

## New cohorts
| Cohort | Tissue | Design | Use |
|---|---|---|---|
| GSE137186 | BM | 4 donors, P1, P5, P10, P15, 2 resequenced samples | Pooled; test-retest; authors report retained immunoregulatory potency |
| GSE275980 | MSC, tissue unstated in the abstract | 3 donors, P2, P6, P10 | Pooled |
| GSE125632 | BM | 5 donors, P2 vs P11 labelled senescent by authors | Pooled; labelled check |
| GSE194298 | BM | 4 donors, p1 vs p4/5, FBS and hPL | Pooled per donor; medium effect |
| GSE119987 | UC | 4 donors to replicative senescence, array, doubling level known | Pooled; early change |
| GSE56362 | BM | Donors at P3, P5, P7 with technical replicates, array | Pooled; test-retest |
| GSE233634 | UC | P3 to P12 in triplicate, no donor stated, counts as one unit | Pooled as one lineage |
| GSE269586 | WJ | GMP production, master bank to P2 or P3 | Not pooled: short-expansion comparison |
| GSE48662 | BM | Irradiated vs control; GVHD responders G1 (5) vs non-responders G2 (3), array | Labelled check; clinical-response question |

## Primary question
Across donors in new pooled cohorts, does the published SenCat score rise with passage? Test: two-sided exact binomial on donors, plus a cohort-level sign-flip test. Log1p weighted score on RNA-seq; rank-weighted form on arrays. Sensitivities: rank form everywhere, weights refit without BMMSC and preadipocytes, excluding GSE56362 (gene-normalized values), and excluding GSE233634.

## Head-to-head
Published SenCat, hUSI, SenMayo, the 7-gene signature of Yang 2023, the 40 AD-MSC biomarkers, Hallmark E2F and G2M (inverted), and CDKN2A, CDKN1A, IL6, GDF15, LMNB1 (inverted) transcripts, all in rank form. Reported: pooled donors rising, median within-donor rho, cohorts with a positive mean. Results on a score's own training cohort are flagged and left out of its pooled count. No single winner is declared unless its lead is clear of the uncertainty.

## Clinical-response question
One directional hypothesis: published SenCat rank score is higher in GVHD non-responders (G2) than responders (G1). Exact permutation, smallest one-sided p 1/56. The outcome will be reported either way, and it would be hypothesis-generating at any p.

## Labelled MSC checks
AUROC is allowed only where the authors define the classes: GSE48662 control vs irradiated, GSE125632 P2 vs P11 senescent. Within-cohort only, with no transported threshold.

## Drift clock (trained model)
Ridge regression of within-donor log2(passage + 1) on within-sample gene ranks, with leave-one-cohort-out validation and all selection inside training cohorts. It estimates passage-like drift and is not a senescence or potency measure. Negative control: passages shuffled within donors in training. It enters the demo only if held-out median within-donor rho is at least 0.8, at least 90% of held-out donors rise, and the shuffled control's median rho stays below 0.3.

## Robustness
Binomial thinning of counts, random gene dropout, technical test-retest, leave-one-sample-out normalization, medium effect, and absolute-score comparability across cohorts. These set guard thresholds in the tool from measurement instead of judgement.
"""
amend_path = REPORTS / 'ANALYSIS_AMENDMENT_1.md'
amend_path.write_text(AMENDMENT, encoding='utf-8')
print('Amendment sha256', sg.sha256(amend_path))

## Download and parse each cohort
Each cohort loads inside its own try block. A failure is recorded and doesn't stop the others.

In [ ]:
UA = {'User-Agent': 'SenGuard-MSC-audit/1.0'}
GEO = 'https://ftp.ncbi.nlm.nih.gov/geo'
CGI = 'https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi'

def fetch(url, name=None):
    dest = RAWX / (name or url.rsplit('/', 1)[-1])
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    for attempt in range(5):
        try:
            if attempt < 3:
                with requests.get(url, headers=UA, stream=True, timeout=300) as r:
                    r.raise_for_status()
                    with open(dest, 'wb') as f:
                        for c in r.iter_content(1 << 20):
                            f.write(c)
            else:
                subprocess.run(['curl', '-fsSL', '--http1.1', '-A', UA['User-Agent'], '-o', str(dest), url], check=True, timeout=600)
            if dest.stat().st_size > 0:
                if dest.suffix == '.gz':
                    with gzip.open(dest, 'rb') as f:
                        while f.read(1 << 20):
                            pass
                return dest
        except Exception as e:
            print('  retry', dest.name, type(e).__name__, str(e)[:90])
        time.sleep(4 * (attempt + 1))
    raise RuntimeError(f'Could not download {url}')

def sdir(acc):
    return f'{GEO}/series/{acc[:-3]}nnn/{acc}'

def parse_matrix(path):
    meta, chars = {}, []
    rows = []
    in_table = False
    with gzip.open(path, 'rt', encoding='utf-8', errors='replace') as f:
        for line in f:
            if line.startswith('!series_matrix_table_begin'):
                in_table = True
            elif line.startswith('!series_matrix_table_end'):
                break
            elif in_table:
                rows.append(line.rstrip('\n').replace('"', '').split('\t'))
            elif line.startswith('!Sample_'):
                parts = line.rstrip('\n').replace('"', '').split('\t')
                if parts[0] == '!Sample_characteristics_ch1':
                    chars.append(parts[1:])
                else:
                    meta[parts[0][8:]] = parts[1:]
    expr = pd.DataFrame(rows[1:], columns=rows[0]).set_index('ID_REF').apply(pd.to_numeric, errors='coerce').T
    m = pd.DataFrame({'sample_id': meta['geo_accession'], 'title': meta['title']})
    for col in chars:
        key = col[0].split(':')[0].strip().lower()
        m[key] = [c.split(':', 1)[1].strip() if ':' in c else c for c in col]
    return expr, m.set_index('sample_id')

def probe_symbols(gpl):
    annot = f'{GEO}/platforms/{gpl[:-3]}nnn/{gpl}/annot/{gpl}.annot.gz'
    soft = f'{GEO}/platforms/{gpl[:-3]}nnn/{gpl}/soft/{gpl}_family.soft.gz'
    try:
        path = fetch(annot, f'{gpl}.annot.gz')
    except Exception:
        path = fetch(soft, f'{gpl}_family.soft.gz')
    rows, header, on = [], None, False
    with gzip.open(path, 'rt', encoding='utf-8', errors='replace') as f:
        for line in f:
            if line.startswith('!platform_table_begin'):
                on = True
                continue
            if line.startswith('!platform_table_end'):
                break
            if on:
                parts = line.rstrip('\n').split('\t')
                if header is None:
                    header = parts
                elif len(parts) == len(header):
                    rows.append(parts)
    t = pd.DataFrame(rows, columns=header)
    col = next((c for c in t.columns if c.lower() in ('gene symbol', 'gene_symbol', 'genesymbol', 'symbol', 'gene_name', 'genename')), None)
    if col is None and 'gene_assignment' in t.columns:
        t['sym'] = t.gene_assignment.str.split('//').str[1].str.strip()
        col = 'sym'
    if col is None:
        raise RuntimeError(f'{gpl}: no symbol column in {list(t.columns)[:15]}')
    s = t.set_index('ID')[col].astype(str).str.strip()
    s = s[(s != '') & (s != 'nan') & (s != '---') & ~s.str.contains('///')]
    return s

ens2sym_tbl = pd.read_csv(PROC / f'{ASC}_matrix_as_deposited.tsv.gz', sep='\t', usecols=['ensembl', 'symbol'], dtype=str).dropna().drop_duplicates('ensembl').set_index('ensembl').symbol
sym2ens_tbl = pd.Series(ens2sym_tbl.index, index=ens2sym_tbl.values)
sym2ens_tbl = sym2ens_tbl[~sym2ens_tbl.index.duplicated(keep=False)]

def to_sym(expr):
    """Samples x genes to samples x symbols; Ensembl ids are mapped through annotation, keeping the highest-mean gene per symbol."""
    cols = pd.Index(expr.columns).astype(str)
    if pd.Series(cols).str.match(r'^ENSG').mean() > 0.5:
        c = cols.str.replace(r'\.\d+$', '', regex=True)
        e = expr.copy(); e.columns = c
        return sg.to_symbols(e.loc[:, ~e.columns.duplicated()], pd.DataFrame({'symbol': ens2sym_tbl.reindex(c.unique())}, index=c.unique()))
    e = expr.loc[:, ~pd.Index(cols).duplicated()]
    return e

def to_ens(expr):
    cols = pd.Index(expr.columns).astype(str)
    if pd.Series(cols).str.match(r'^ENSG').mean() > 0.5:
        e = expr.copy(); e.columns = cols.str.replace(r'\.\d+$', '', regex=True)
        return e.loc[:, ~e.columns.duplicated()]
    m = sym2ens_tbl.reindex(cols)
    e = expr.loc[:, m.notna().to_numpy()].copy()
    e.columns = m.dropna().to_numpy()
    return e.loc[:, ~e.columns.duplicated()]

COHORTS, LOAD_LOG = {}, []
def add(key, expr, meta, kind, tissue, pooled, role, note=''):
    """kind: counts, normcounts, fpkm, log_array."""
    meta = meta.loc[expr.index].copy()
    if not {'subject', 'passage', 'unit'} <= set(meta.columns):
        raise ValueError(f'{key}: meta needs subject, passage, unit; has {list(meta.columns)}')
    meta['passage'] = meta.passage.astype(float)
    COHORTS[key] = dict(expr=expr, meta=meta, kind=kind, tissue=tissue, pooled=pooled, role=role, note=note)
    LOAD_LOG.append({'cohort': key, 'status': 'loaded', 'samples': len(expr), 'genes': expr.shape[1], 'subjects': meta.subject.nunique(), 'units': meta.unit.nunique(), 'kind': kind, 'tissue': tissue, 'pooled': pooled, 'note': note})
    print(f'{key}: {expr.shape} subjects {meta.subject.nunique()} units {meta.unit.nunique()} passages {sorted(meta.passage.unique())[:10]} kind {kind}')

def attempt(key, fn):
    try:
        fn()
    except Exception as e:
        LOAD_LOG.append({'cohort': key, 'status': f'failed: {type(e).__name__}: {str(e)[:220]}'})
        print(key, 'FAILED', type(e).__name__, str(e)[:300])

def geo_samples(acc):
    t = requests.get(CGI, params={'acc': acc, 'targ': 'gsm', 'form': 'text', 'view': 'full'}, headers=UA, timeout=180).text
    rows = []
    for b in re.split(r'\^SAMPLE = ', t)[1:]:
        ch = {c.split(':')[0].strip().lower(): c.split(':', 1)[1].strip() for c in re.findall(r'!Sample_characteristics_ch1 = (.*)', b) if ':' in c}
        rows.append({'gsm': b.split('\n')[0].strip(), 'title': re.findall(r'!Sample_title = (.*)', b)[0].strip(), 'description': ' | '.join(x.strip() for x in re.findall(r'!Sample_description = (.*)', b)), **ch})
    time.sleep(0.5)
    return pd.DataFrame(rows)

# Already audited cohorts
def _asc():
    m = sg.load_manifest(); a = m[(m.study_accession == ASC)]
    e = pd.read_csv(PROC / f'{ASC}_norm_counts.tsv.gz', sep='\t', index_col=0).T.loc[a.index]
    meta = pd.DataFrame({'subject': a.donor.astype(str), 'unit': a.donor.astype(str), 'passage': a.passage}, index=a.index)
    add(ASC, e, meta, 'normalized', 'AD', False, 'seen')
attempt(ASC, _asc)
def _bm():
    m = sg.load_manifest(); a = m[(m.study_accession == BM) & m.use_in_mrna_analysis]
    e = pd.read_csv(PROC / f'{BM}_tpm_ensembl.tsv.gz', sep='\t', index_col=0).T.loc[a.index]
    ids = a.candidate_biological_id.astype(int).astype(str)
    add(BM, e, pd.DataFrame({'subject': ids, 'unit': ids, 'passage': a.passage}, index=a.index), 'fpkm', 'BM', False, 'seen', 'Donors inferred')
attempt(BM, _bm)

def _137():
    p = fetch(f'{sdir("GSE137186")}/suppl/GSE137186_raw.counts.txt.gz')
    t = pd.read_csv(p, sep='\t', header=None, dtype=str, low_memory=False)
    head = t.iloc[:4]
    body = t.iloc[4:]
    body = body[~body[1].astype(str).str.startswith('__')]
    expr = body.set_index(1).iloc[:, 1:].apply(pd.to_numeric, errors='coerce').T
    names = [f'S{i}' for i in head.iloc[1, 2:]]
    expr.index = names
    meta = pd.DataFrame({'subject': head.iloc[2, 2:].to_numpy(), 'unit': head.iloc[2, 2:].to_numpy(), 'passage': head.iloc[3, 2:].astype(float).to_numpy()}, index=names)
    expr = expr.loc[:, ~expr.columns.duplicated()]
    add('GSE137186', expr, meta, 'counts', 'BM', True, 'new', 'Authors report retained immunoregulatory potency')
attempt('GSE137186', _137)

def _275():
    p = fetch(f'{sdir("GSE275980")}/suppl/GSE275980_raw_count.xlsx')
    t = pd.read_excel(p)
    t = t.drop_duplicates(t.columns[0]).set_index(t.columns[0])
    t.columns = [c.replace('P2C2', 'P2-C2') for c in t.columns]
    expr = t.apply(pd.to_numeric, errors='coerce').fillna(0).T
    parts = pd.Series(expr.index).str.extract(r'P(\d+)-C(\d)')
    meta = pd.DataFrame({'subject': ('C' + parts[1]).to_numpy(), 'unit': ('C' + parts[1]).to_numpy(), 'passage': parts[0].astype(float).to_numpy()}, index=expr.index)
    add('GSE275980', expr, meta, 'counts', 'MSC', True, 'new', 'Gene symbols only; C1-C3 are the three donors in the series title')
attempt('GSE275980', _275)

def _125():
    p = fetch(f'{sdir("GSE125632")}/suppl/GSE125632_FPKM_iMSC_manuscript.txt.gz')
    t = pd.read_csv(p, sep='\t', low_memory=False, on_bad_lines='skip', dtype=str)
    t = t[t.EnsemblID.astype(str).str.startswith('ENSG')].drop_duplicates('EnsemblID').set_index('EnsemblID')
    cols = [f'Sample_{i}' for i in range(1, 21)]
    expr = t[cols].apply(pd.to_numeric, errors='coerce').fillna(0).T
    g = geo_samples('GSE125632')
    meta = pd.DataFrame({'title': g.title.to_numpy(), 'subject': g.donor.to_numpy(), 'passage': g.passage.str.replace('P', '').astype(float).to_numpy()}, index=cols)
    meta['arm'] = np.where(meta.title.str.startswith('iMSC'), 'iMSC', 'MSC')
    meta['gender'] = g.gender.to_numpy()
    y = t.reindex(['ENSG00000129824'])[cols].apply(pd.to_numeric, errors='coerce').iloc[0]
    agree = float(((y > 1).to_numpy() == (meta.gender == 'Male').to_numpy()).mean())
    print('  GSE125632 column order check: RPS4Y1 agrees with GEO sex in', f'{agree:.0%}', 'of samples')
    keep = meta.arm == 'MSC'
    m = meta[keep].assign(unit=lambda d: d.subject)
    add('GSE125632', expr.loc[keep], m, 'fpkm', 'BM', True, 'new', f'Column order to GSM inferred; sex check agreement {agree:.0%}')
    im = meta[~keep].assign(unit=lambda d: d.subject)
    add('GSE125632_iMSC', expr.loc[~keep], im, 'fpkm', 'iPSC-derived', False, 'new', 'Reported separately')
attempt('GSE125632', _125)

def _194():
    p = fetch(f'{sdir("GSE194298")}/suppl/GSE194298_Norm_counts_RNAseq_hPL_vs_FBS.csv.gz')
    t = pd.read_csv(p, index_col=0)
    t.columns = [re.search(r'X(\d+)', c).group(1) for c in t.columns]
    g = geo_samples('GSE194298')
    g['num'] = g.title.str.extract(r'_(\d+)$')
    g = g.set_index('num').loc[t.columns]
    expr = t.T
    medium = g.treatment.str.strip().to_numpy()
    meta = pd.DataFrame({'donor': g.donor.to_numpy(), 'medium': medium, 'passage': g.passage.map({'p1': 1.0, 'p4/p5': 4.5}).to_numpy()}, index=t.columns)
    meta['subject'] = meta.donor + '_' + meta.medium
    meta['unit'] = meta.donor
    add('GSE194298', expr, meta, 'normcounts', 'BM', True, 'new', 'Medium arms averaged per donor; mBG arm has no p1')
attempt('GSE194298', _194)

def _233():
    p = fetch(f'{sdir("GSE233634")}/suppl/GSE233634_gene_FPKM.txt.gz')
    t = pd.read_csv(p, sep='\t', low_memory=False).set_index('geneID')
    plane = [c for c in t.columns if re.fullmatch(r'P\d+_\d', c)]
    bio = [c for c in t.columns if re.fullmatch(r'BioR\d_\d', c)]
    expr = t[plane].apply(pd.to_numeric, errors='coerce').T
    meta = pd.DataFrame({'passage': pd.Series(plane).str.extract(r'P(\d+)_')[0].astype(float).to_numpy(), 'subject': 'lineage', 'unit': 'lineage', 'rep': pd.Series(plane).str.extract(r'_(\d)')[0].to_numpy()}, index=plane)
    add('GSE233634', expr, meta, 'fpkm', 'UC', True, 'new', 'No donor stated; one unit')
    COHORTS['GSE233634']['bioreactor'] = t[bio].apply(pd.to_numeric, errors='coerce').T
attempt('GSE233634', _233)

def _269():
    p = fetch(f'{sdir("GSE269586")}/suppl/GSE269586_all_counts_merged.txt.gz')
    t = pd.read_csv(p, sep='\t', low_memory=False).set_index('id')
    g = geo_samples('GSE269586')
    g = g[g.title.str.contains('RNAseq')].copy()
    g['col'] = 'raw_' + g.description.str.split(' \\| ').str[0]
    missing = [c for c in g.col if c not in t.columns]
    if missing:
        raise RuntimeError(f'columns missing: {missing[:4]}')
    expr = t[g.col.tolist()].apply(pd.to_numeric, errors='coerce').fillna(0).T
    expr.index = g.gsm.to_numpy()
    donor = g.donor.str.replace('donor ', '').to_numpy()
    meta = pd.DataFrame({'subject': donor, 'unit': donor, 'passage': g.passage.astype(float).to_numpy()}, index=g.gsm.to_numpy())
    add('GSE269586', expr, meta, 'counts', 'WJ', False, 'new', 'GMP production; short expansion; not pooled')
attempt('GSE269586', _269)

def _119():
    p = fetch(f'{sdir("GSE119987")}/matrix/GSE119987_series_matrix.txt.gz')
    expr, m = parse_matrix(p)
    sym = probe_symbols('GPL571')
    keep = [c for c in expr.columns if c in sym.index]
    e = expr[keep]
    e = sg.to_symbols(e, pd.DataFrame({'symbol': sym.reindex(keep)}, index=keep))
    meta = pd.DataFrame({'subject': m['donor population name'], 'unit': m['donor population name'].str.replace(r'\.\d+$', '', regex=True), 'passage': m['passage number'].astype(float), 'pdl': pd.to_numeric(m.get('population doubling level'), errors='coerce'), 'batch': m.get('overall batch')}, index=m.index)
    add('GSE119987', e, meta, 'log_array', 'UC', True, 'new', 'Cultured to replicative senescence')
attempt('GSE119987', _119)

def _563():
    p = fetch(f'{sdir("GSE56362")}/matrix/GSE56362_series_matrix.txt.gz')
    expr, m = parse_matrix(p)
    sym = probe_symbols('GPL18503')
    keep = [c for c in expr.columns if c in sym.index]
    e = expr[keep]
    e = sg.to_symbols(e, pd.DataFrame({'symbol': sym.reindex(keep)}, index=keep))
    has = m['passage'].astype(str).str.contains(r'\d')
    meta = pd.DataFrame({'subject': m['donor id'], 'unit': m['donor id'], 'passage': m['passage'].astype(str).str.extract(r'(\d+)')[0].astype(float)}, index=m.index)[has]
    e = e.loc[meta.index]
    add('GSE56362', e, meta, 'log_array', 'BM', True, 'new', 'Values look gene-normalized; technical triplicates averaged')
attempt('GSE56362', _563)

def _486():
    p = fetch(f'{sdir("GSE48662")}/matrix/GSE48662-GPL13497_series_matrix.txt.gz')
    expr, m = parse_matrix(p)
    sym = probe_symbols('GPL13497')
    keep = [c for c in expr.columns if c in sym.index]
    e = sg.to_symbols(expr[keep], pd.DataFrame({'symbol': sym.reindex(keep)}, index=keep))
    m['group'] = m.title.str.extract(r'MSC[_-](WT|SEN|G1|G2)')[0]
    m = m[m.group.notna()]
    meta = m[['title', 'group']].copy()
    meta['subject'] = m.title.str.extract(r'-(\d+)')[0].to_numpy()
    meta['unit'] = meta.subject
    meta['passage'] = np.nan
    COHORTS['GSE48662'] = dict(expr=e.loc[m.index], meta=meta, kind='log_array', tissue='BM', pooled=False, role='new', note='Control vs irradiated, and GVHD responder groups')
    LOAD_LOG.append({'cohort': 'GSE48662', 'status': 'loaded', 'samples': len(m), 'genes': e.shape[1], 'kind': 'log_array', 'tissue': 'BM', 'pooled': False, 'note': str(m.group.value_counts().to_dict())})
    print('GSE48662', e.loc[m.index].shape, m.group.value_counts().to_dict())
attempt('GSE48662', _486)
pd.DataFrame(LOAD_LOG).to_csv(RESULTS / 'ext_cohort_load_log.csv', index=False)
print(pd.DataFrame(LOAD_LOG)[['cohort', 'status', 'samples', 'genes', 'subjects', 'units', 'kind']].to_string(index=False))

## Signatures and scoring
Weights and gene lists are fixed. Rank forms are computed for all cohorts, so arrays and RNA-seq can be compared. For RNA-seq the published log1p weighted form is also computed, after the median-of-ratios step where counts allow it.

In [ ]:
pub_tbl = sg.load_sencat_markers()
PUB = pub_tbl.coef
pub_sym = pub_tbl.dropna(subset=['gene']).drop_duplicates('gene').set_index('gene').coef
REFIT = pd.read_csv(RESULTS / 'frozen_weights_refit_no_BMMSC_prAD.csv').set_index('gene_id').coef
refit_sym = REFIT.copy(); refit_sym.index = ens2sym_tbl.reindex(REFIT.index).to_numpy(); refit_sym = refit_sym[refit_sym.index.notna()]; refit_sym = refit_sym[~refit_sym.index.duplicated()]
HUSI = sg.load_husi_weights()
hall, extra = sg.load_gene_sets()
YANG7 = ['PRUNE2', 'DIO2', 'CPA4', 'PRKAA2', 'DMD', 'DDAH1', 'GATA6']
UP40 = 'CACNB4 CRYAB DAW1 EFEMP1 EPAS1 GALNT5 H1-2 KRT34 MELTF MEST MFGE8 MGARP MOXD1 SERPINB2 SERPINE2 SLC43A2 SRGN SVEP1 SYNPO2 TMEM178B'.split()
DOWN40 = 'AURKB BUB1 BUB1B CDC20 DEPDC1 KIF14 KIF18B KIF2C KIFC1 KNL1 LMNB1 NCAPH NUF2 PIMREG PKMYT1 SKA3 TNC TROAP TTK UBE2C'.split()

def rank_weighted(sym, w):
    shared = [g for g in w.index if g in sym.columns]
    wr = stats.rankdata(w.loc[shared].to_numpy())
    r = np.apply_along_axis(stats.rankdata, 1, sym[shared].fillna(0).to_numpy(dtype=float))
    wc, rc = wr - wr.mean(), r - r.mean(axis=1, keepdims=True)
    return pd.Series((rc @ wc) / (np.linalg.norm(rc, axis=1) * np.linalg.norm(wc)), index=sym.index), len(shared)

def primary_input(c):
    e = to_ens(c['expr'])
    if c['kind'] == 'counts':
        n = sg.mor_normalize(sg.author_gene_filter(e))[0]
    elif c['kind'] == 'fpkm':
        n = sg.scale_to_million(e)
    else:
        n = e
    return n

SCORES, COVER = {}, []
for key, c in COHORTS.items():
    sym = to_sym(c['expr'])
    sym = sym.fillna(sym.median()).fillna(0) if c['kind'] == 'log_array' else sym.fillna(0)
    c['sym'] = sym
    out = {}
    if c['kind'] != 'log_array':
        n = primary_input(c)
        c['norm'] = n
        out['SenCat published'], u1 = sg.weighted_log1p_score(n, PUB)
        out['SenCat refit, lineage excluded'], _ = sg.weighted_log1p_score(n, REFIT)
    else:
        out['SenCat published'], u1 = rank_weighted(sym, pub_sym)
        out['SenCat refit, lineage excluded'], _ = rank_weighted(sym, refit_sym)
    out['SenCat published, rank form'], u2 = rank_weighted(sym, pub_sym)
    out['hUSI'], u3 = sg.husi_raw(sym, HUSI)
    ranks = sym.rank(axis=1, pct=True)
    for nm, genes, sign in [('SenMayo', extra['SenMayo'], 1), ('Yang 2023 7 genes', YANG7, 1), ('Hallmark E2F (inverted)', hall['HALLMARK_E2F_TARGETS'], -1), ('Hallmark G2M (inverted)', hall['HALLMARK_G2M_CHECKPOINT'], -1)]:
        have = [g for g in genes if g in ranks.columns]
        out[nm] = sign * ranks[have].mean(axis=1)
        COVER.append({'cohort': key, 'score': nm, 'genes_present': len(have), 'genes_total': len(genes)})
    hu, hd = [g for g in UP40 if g in ranks.columns], [g for g in DOWN40 if g in ranks.columns]
    out['AD-MSC 40 biomarkers'] = ranks[hu].mean(axis=1) - ranks[hd].mean(axis=1)
    for g, sign in [('CDKN2A', 1), ('CDKN1A', 1), ('IL6', 1), ('GDF15', 1), ('LMNB1', -1)]:
        if g in ranks.columns:
            out[g + (' (inverted)' if sign < 0 else '')] = sign * ranks[g]
    COVER.append({'cohort': key, 'score': 'SenCat published', 'genes_present': int(u1), 'genes_total': len(PUB)})
    COVER.append({'cohort': key, 'score': 'hUSI', 'genes_present': int(u3), 'genes_total': len(HUSI)})
    SCORES[key] = pd.DataFrame(out)
pd.DataFrame(COVER).to_csv(RESULTS / 'ext_signature_coverage.csv', index=False)
cov = pd.DataFrame(COVER).assign(pct=lambda d: (100 * d.genes_present / d.genes_total).round(1)).pivot(index='cohort', columns='score', values='pct')
print(cov.to_string())

husi_meta = pd.read_csv(EXT / 'HUSI/Data/trainSet_meta843.csv', low_memory=False, encoding='latin-1')
all_text = husi_meta.astype(str).agg(' '.join, axis=1)
husi_overlap = {k.split('_')[0]: int(all_text.str.contains(k.split('_')[0]).sum()) for k in COHORTS}
print('hUSI training rows by cohort accession:', husi_overlap)
CIRCULAR = {'hUSI': {k for k, v in husi_overlap.items() if v > 0}, 'AD-MSC 40 biomarkers': {ASC}, 'Yang 2023 7 genes': {BM}}
json.dump({'husi_rows_by_accession': husi_overlap, 'circular': {k: sorted(v) for k, v in CIRCULAR.items()}}, open(RESULTS / 'ext_circularity.json', 'w'), indent=1)

long = []
for key, s in SCORES.items():
    m = COHORTS[key]['meta']
    x = s.join(m[['subject', 'unit', 'passage']]).rename_axis('sample').reset_index().melt(['sample', 'subject', 'unit', 'passage'], var_name='score_name', value_name='score')
    x.insert(0, 'cohort', key)
    long.append(x)
pd.concat(long).to_csv(RESULTS / 'ext_scores_long.csv', index=False)

SenGuard-MSC · Notebook 08 — Score forecast: independent test on cohorts never used

The gene-level virtual passage failed its gate in notebook 07, but the score forecast passed that one criterion. Passing one criterion out of several, after seeing it, can't justify a claim. So the forecast is frozen here and tested on cohorts that nothing in this project has touched.

In [ ]:
AMENDMENT3 = """# Analysis protocol amendment 3: independent test of the score forecast

Fixed on 2026-11-01, after notebook 07, before any cohort below was scored.

## Frozen model
virtual_passage_model.json from notebook 07, model "final": per-gene drift across 8 cohorts, 4 significant digits. Forecast of the SenCat rank score at a later passage = rank score of (earliest profile + drift x change in log2(passage + 1)). No refitting.

## Independent cohorts, chosen from GEO metadata alone
- GSE39035: bone marrow, 8 donors, P4 and P8, array.
- GSE139073: bone marrow, 6 donors with both P3 and P6 at rest (BM126, BM127, BM285, BM293, BM324, BM784), RNA-seq. Stimulated samples are not used.
- GSE9593: bone marrow, 3 donors, P2 to P7, P8 or P11, array.
If a cohort can't be parsed, it is reported as failed and the rule applies to what remains, with at least 2 cohorts.

## Gate for the interface
All must hold:
1. Absolute error of the forecast is lower than carry-forward in at least 70% of donors, pooled.
2. Median error is lower in every available cohort except at most one.
3. Negative control: drift learned from passages shuffled within donors beats carry-forward in at most 60% of donors, on average.
4. Prediction range: the 80% range set from leave-one-cohort-out errors in notebook 07 contains the actual score in at least 70% of donors.

## What this claims if it passes
A forecast of how a transcriptomic signature moves with passage. Not senescence confirmation, growth capacity, potency or safety.
"""
(REPORTS / 'ANALYSIS_AMENDMENT_3.md').write_text(AMENDMENT3, encoding='utf-8')
print('Amendment 3 sha256', sg.sha256(REPORTS / 'ANALYSIS_AMENDMENT_3.md'))
model = json.load(open(RESULTS / 'virtual_passage_model.json'))
print('Frozen model sha256', sg.sha256(RESULTS / 'virtual_passage_model.json'))
units_dev = pd.read_csv(RESULTS / 'virtual_passage_units.csv')
RANGE80 = float(np.percentile(units_dev.err_score_model, 80))
print('80% error range from leave-one-cohort-out:', round(RANGE80, 4), 'on', len(units_dev), 'donors')
GENES = model['genes']; BETA = np.array(model['beta']['final']); SIG = pd.Series(model['signature'])

## Load the independent cohorts

In [ ]:
TEST, LOG = {}, []
def attempt(key, fn):
    try:
        fn()
    except Exception as e:
        LOG.append({'cohort': key, 'status': f'failed: {type(e).__name__}: {str(e)[:240]}'})
        print(key, 'FAILED', type(e).__name__, str(e)[:300])

def add_test(key, sym, meta, kind, note=''):
    meta = meta.loc[sym.index]
    TEST[key] = dict(sym=sym, meta=meta, kind=kind)
    LOG.append({'cohort': key, 'status': 'loaded', 'samples': len(sym), 'genes': sym.shape[1], 'donors': meta.subject.nunique(), 'kind': kind, 'note': note})
    print(key, sym.shape, 'donors', meta.subject.nunique(), sorted(meta.passage.unique()))

def _39035():
    expr, m = parse_matrix(fetch(f'{sdir("GSE39035")}/matrix/GSE39035_series_matrix.txt.gz'))
    gpl = re.search(r'GPL\d+', gzip.open(RAWX / 'GSE39035_series_matrix.txt.gz', 'rt', errors='replace').read(200000)).group(0)
    sym = probe_symbols(gpl)
    keep = [c for c in expr.columns if c in sym.index]
    e = sg.to_symbols(expr[keep], pd.DataFrame({'symbol': sym.reindex(keep)}, index=keep))
    meta = pd.DataFrame({'subject': m['subject id'].astype(str), 'passage': m['passage'].astype(float)}, index=m.index)
    add_test('GSE39035', e.fillna(e.median()).fillna(0), meta, 'log_array', f'Platform {gpl}')
attempt('GSE39035', _39035)

def _139073():
    t = pd.read_csv(fetch(f'{sdir("GSE139073")}/suppl/GSE139073_raw_counts.txt.gz'), sep='\t', low_memory=False)
    print('  columns', list(t.columns)[:6], '| first ids', t.iloc[:2, 0].tolist())
    g = geo_samples('GSE139073')
    g['s'] = g.title.str.extract(r'(S\d+)')[0]
    donors = ['BM126', 'BM127', 'BM285', 'BM293', 'BM324', 'BM784']
    g = g[g.donor.isin(donors) & (g.stimulation == 'REST') & g.passage.isin(['P3', 'P6'])]
    print('  selected', g[['donor', 'passage', 's']].values.tolist())
    if isinstance(t.index, pd.RangeIndex):   # this file's header is one field shorter than its rows, so ids are already the index
        t = t.set_index(t.columns[0])
    t = t[~t.index.duplicated()]
    print('  gene ids', t.index[:3].tolist(), '| rows', len(t))
    cols = {}
    for r in g.itertuples():
        hit = [c for c in t.columns if re.search(rf'(^|[^0-9]){r.s[1:]}([^0-9]|$)', str(c))]
        if len(hit) != 1:
            raise RuntimeError(f'{r.s}: matched {hit}; columns {list(t.columns)[:8]}')
        cols[hit[0]] = r.s
    x = t[list(cols)].rename(columns=cols).apply(pd.to_numeric, errors='coerce').fillna(0).T
    meta = pd.DataFrame({'subject': g.donor.to_numpy(), 'passage': g.passage.str[1:].astype(float).to_numpy()}, index=g.s.to_numpy())
    add_test('GSE139073', to_sym(x).fillna(0), meta, 'counts', 'Resting samples')
attempt('GSE139073', _139073)

def _9593():
    expr, m = parse_matrix(fetch(f'{sdir("GSE9593")}/matrix/GSE9593_series_matrix.txt.gz'))
    gpl = re.search(r'GPL\d+', gzip.open(RAWX / 'GSE9593_series_matrix.txt.gz', 'rt', errors='replace').read(200000)).group(0)
    sym = probe_symbols(gpl)
    keep = [c for c in expr.columns if c in sym.index]
    e = sg.to_symbols(expr[keep], pd.DataFrame({'symbol': sym.reindex(keep)}, index=keep))
    t = m.title.str.extract(r'donor(\d)')[0]
    p = m.title.str.extract(r'-P(\d+)')[0].astype(float)
    meta = pd.DataFrame({'subject': 'donor' + t, 'passage': p}, index=m.index)
    add_test('GSE9593', e.fillna(e.median()).fillna(0), meta, 'array', f'Platform {gpl}')
attempt('GSE9593', _9593)
pd.DataFrame(LOG).to_csv(RESULTS / 'forecast_test_load_log.csv', index=False)
print(pd.DataFrame(LOG).to_string(index=False, max_colwidth=90))

## Test [measured]

In [ ]:
tlog = lambda p: np.log2(np.asarray(p, float) + 1)
def rank_form(sym):
    use = [g for g in GENES if g in sym.columns]
    return sym[use].rank(axis=1, pct=True), use

def evaluate(beta_full):
    out = []
    bs = pd.Series(beta_full, index=GENES)
    for key, c in TEST.items():
        R, use = rank_form(c['sym'])
        sig = SIG.reindex(use).dropna()
        s_idx = [use.index(g) for g in sig.index]
        wr = stats.rankdata(sig.to_numpy())
        score = lambda x: float(np.corrcoef(stats.rankdata(np.asarray(x)[s_idx]), wr)[0, 1])
        b = bs[use].to_numpy()
        d = R.join(c['meta']).groupby(['subject', 'passage']).mean(numeric_only=True).reset_index()
        for sub, g in d.groupby('subject'):
            g = g.sort_values('passage')
            if g.passage.nunique() < 2:
                continue
            X = g[use].to_numpy(float)
            x0, xL = X[0], X[-1]
            dt = tlog(g.passage.iloc[-1]) - tlog(g.passage.iloc[0])
            s0, sT, sF = score(x0), score(xL), score(x0 + b * dt)
            out.append({'cohort': key, 'donor': sub, 'from': g.passage.iloc[0], 'to': g.passage.iloc[-1], 'score_start': s0, 'score_actual': sT, 'score_forecast': sF, 'err_forecast': abs(sF - sT), 'err_carry': abs(s0 - sT), 'genes': len(use), 'signature_genes': len(s_idx)})
    return pd.DataFrame(out)

res = evaluate(BETA)
res['forecast_better'] = res.err_forecast < res.err_carry
res['in_range'] = res.err_forecast <= RANGE80
res['actual_rose'] = res.score_actual > res.score_start
res.to_csv(RESULTS / 'forecast_test_donors.csv', index=False)
print(res.round(4).to_string(index=False))
# Very small errors are checked before they are believed: leakage would show up as a cohort shared with training, or as identical profiles
print('Test cohorts also used in training:', sorted(set(TEST) & {'GSE309666', 'GSE178514', 'GSE137186', 'GSE275980', 'GSE194298', 'GSE233634', 'GSE119987', 'GSE56362'}))
for k, c in TEST.items():
    R, use = rank_form(c['sym'])
    print(k, 'correlation of each sample with the training drift vector, spread:', pd.Series(R.sub(R.mean()).to_numpy() @ pd.Series(BETA, index=GENES)[use].to_numpy()).round(1).describe()[['min', 'max']].to_dict())
    print(k, 'smallest change between two samples of one donor, mean absolute rank:', round(float(min(np.abs(R.iloc[i] - R.iloc[j]).mean() for i in range(len(R)) for j in range(i + 1, len(R)))), 4))
by = res.groupby('cohort').agg(donors=('donor', 'size'), err_forecast=('err_forecast', 'median'), err_carry=('err_carry', 'median'), better=('forecast_better', 'sum'), in_range=('in_range', 'sum'), rose=('actual_rose', 'sum'))
print(by.round(3).to_string())

# Negative control: drift learned from training cohorts with passages shuffled inside donors
TRAIN = ['GSE309666', 'GSE178514', 'GSE137186', 'GSE275980', 'GSE194298', 'GSE233634', 'GSE119987', 'GSE56362']
TR = {}
for k in TRAIN:
    if k not in COHORTS:
        continue
    c = COHORTS[k]
    use = [g for g in GENES if g in c['sym'].columns]
    R = c['sym'][use].rank(axis=1, pct=True).reindex(columns=GENES)
    d = R.join(c['meta'][['subject', 'passage']]).groupby(['subject', 'passage'], as_index=False).mean(numeric_only=True)
    TR[k] = [(tlog(g.passage), g[GENES].to_numpy(np.float32)) for _, g in d.groupby('subject') if g.passage.nunique() >= 2]
rng = np.random.default_rng(sg.SEED)
def learn(shuffle):
    per = []
    for k, subs in TR.items():
        bs = []
        for t, X in subs:
            tt = rng.permutation(t) if shuffle else t
            tc = tt - tt.mean()
            if (tc ** 2).sum() > 0:
                bs.append(np.nan_to_num((tc[None, :] @ (X - np.nanmean(X, axis=0))).ravel() / (tc ** 2).sum()))
        per.append(np.mean(bs, axis=0))
    return np.mean(per, axis=0)
check = learn(False)
print('Relearned drift vs frozen drift, correlation:', round(float(np.corrcoef(check, BETA)[0, 1]), 5))
neg = []
for rep in range(30):
    r = evaluate(learn(True))
    neg.append(float((r.err_forecast < r.err_carry).mean()))
gate = {
    'donors': len(res), 'cohorts_loaded': len(by),
    'share_better': float(res.forecast_better.mean()),
    'cohorts_better': int((by.err_forecast < by.err_carry).sum()),
    'negative_control_share': float(np.mean(neg)),
    'range80': RANGE80, 'coverage': float(res.in_range.mean()),
    'median_err_forecast': float(res.err_forecast.median()), 'median_err_carry': float(res.err_carry.median()),
    'actual_rose': int(res.actual_rose.sum()),
    'binom_better_p': float(stats.binomtest(int(res.forecast_better.sum()), len(res)).pvalue),
}
gate['passes'] = bool(len(by) >= 2 and gate['share_better'] >= 0.7 and gate['cohorts_better'] >= len(by) - 1 and gate['negative_control_share'] <= 0.6 and gate['coverage'] >= 0.7)
print(json.dumps(gate, indent=1))
json.dump(gate, open(RESULTS / 'forecast_test_gate.json', 'w'), indent=1)
(REPORTS / 'FORECAST_TEST_RESULTS.md').write_text('# Score forecast: independent test\n\nGate: ' + json.dumps(gate, indent=1) + '\n\nBy cohort:\n' + by.round(3).to_string() + '\n\nBy donor:\n' + res.round(3).to_string(index=False) + '\n', encoding='utf-8')

# Long series in GSE9593 donor 1: forecast every later passage from the first sample
if 'GSE9593' in TEST:
    c = TEST['GSE9593']
    R, use = rank_form(c['sym'])
    sig = SIG.reindex(use).dropna(); s_idx = [use.index(g) for g in sig.index]; wr = stats.rankdata(sig.to_numpy())
    score = lambda x: float(np.corrcoef(stats.rankdata(np.asarray(x)[s_idx]), wr)[0, 1])
    b = pd.Series(BETA, index=GENES)[use].to_numpy()
    g = R.join(c['meta']); g = g[g.subject == 'donor1'].sort_values('passage')
    X = g[use].to_numpy(float)
    series = [{'passage': float(p), 'actual': score(x), 'forecast': score(X[0] + b * (tlog(p) - tlog(g.passage.iloc[0]))), 'carry': score(X[0])} for p, x in zip(g.passage, X)]
    print(pd.DataFrame(series).round(3).to_string(index=False))
    pd.DataFrame(series).to_csv(RESULTS / 'forecast_test_long_series.csv', index=False)